# Simulation Complexity Analysis
## Brain States · LZc · PCI Casali-like

**TVBToolkit — standalone analysis notebook**

This notebook demonstrates the three complexity/dynamics analyses available for simulation outputs:

1. **Brain States** — phase-coherence clustering of firing rates or BOLD
2. **LZc** — multichannel Lempel-Ziv complexity (corrected preprocessing)
3. **PCI Casali-like** — perturbational complexity index

---

### A note on modality: firing rates vs BOLD

Before running any analysis you need to decide which signal you are working with.  
The two modalities need **completely different preprocessing** — using the wrong pipeline gives physically meaningless results.

| Signal | Source | Sampling | Frequency content | Use pipeline |
|--------|--------|----------|------------------|--------------|
| fMRI BOLD | empirical or `bold_from_firing_rates()` | TR 1–3 s | 0.01–0.20 Hz | `"standard"` or `"brain_act_legacy"` |
| Firing rates | AdEx SNN / mean-field (5 ms bins) | 5 ms | 2–80 Hz | `"firing_rate"`, `bandpass_hz=(2,80)` |
| Firing rates | TVB whole-brain (raw, 0.1 ms) | 0.1 ms | 2–80 Hz (local) or 0.05–1 Hz (network) | `"firing_rate"` with appropriate band |

**Rule of thumb:** if your signal looks smooth and slow (seconds-scale oscillations), use BOLD pipelines.  
If your signal shows fast oscillations (milliseconds to hundreds of milliseconds), use `"firing_rate"`.

---
## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial.distance import squareform

# TVBToolkit imports
from tvbtoolkit.analysis.brain_states import (
    phase_patterns,
    cluster_brain_states,
    summarize_brain_states,
    sfc_sort_centroids,
    centers_to_matrices,
)
from tvbtoolkit.analysis.spectral import (
    psd_per_region,
    dominant_frequency,
    phase_coherence_validity,
)
from tvbtoolkit.complexity.measures import (
    lzc_multichannel,
    pci_casali_like,
)

np.random.seed(42)
print("Imports OK")

---
## 1. Synthetic Data — Three Conditions

We create synthetic firing-rate data mimicking three conditions:  
- **CNT** (healthy controls): Strong, coherent gamma oscillations (40 Hz) across regions — organized network activity  
- **MCS** (minimally conscious state): Moderate gamma, partial coherence — intermediate structure  
- **UWS** (unresponsive wakefulness): Weak oscillation, high noise — less organized activity  

Signal parameters match TVBToolkit's AdEx SNN default output:  
- `dt_ms = 5.0 ms` (5 ms bin width from `prepare_population_rates`)  
- Duration: 12 000 ms (12 s) — 2400 samples  
- n_regions = 10

In [ ]:
DT_MS    = 5.0       # ms per sample  (AdEx SNN default bin width)
DURATION = 12_000    # ms
N_SAMPLES = int(DURATION / DT_MS)   # 2400
N_REGIONS = 10

t_s = np.arange(N_SAMPLES) * DT_MS / 1000.0  # time in seconds

def make_firing_rates(gamma_hz, coherence, noise_amp, seed=0):
    """Synthetic firing-rate traces: shared gamma oscillation + noise."""
    rng = np.random.default_rng(seed)
    # Shared 40 Hz oscillation with region-specific phase offsets
    phase_offsets = rng.uniform(0, 2 * np.pi * (1 - coherence), N_REGIONS)
    base = np.stack(
        [np.sin(2 * np.pi * gamma_hz * t_s + phi) for phi in phase_offsets],
        axis=1,
    )  # (time, regions)
    noise = rng.normal(0, noise_amp, (N_SAMPLES, N_REGIONS))
    rates = 20 + 10 * base + noise  # mean 20 Hz, modulation 10 Hz
    return rates.clip(0)  # firing rates can't be negative

# CNT: strong coherent 40 Hz gamma, low noise
rates_cnt = make_firing_rates(gamma_hz=40, coherence=0.95, noise_amp=2.0, seed=1)
# MCS: moderate 40 Hz gamma, moderate noise
rates_mcs = make_firing_rates(gamma_hz=40, coherence=0.55, noise_amp=5.0, seed=2)
# UWS: weak 40 Hz gamma, high noise — dominated by random fluctuations
rates_uws = make_firing_rates(gamma_hz=40, coherence=0.10, noise_amp=10.0, seed=3)

fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
for ax, rates, label, col in zip(
    axes,
    [rates_cnt, rates_mcs, rates_uws],
    ['CNT (healthy)', 'MCS', 'UWS'],
    ['steelblue', 'darkorange', 'firebrick'],
):
    ax.plot(t_s[:500], rates[:500, 0], color=col, lw=0.8, label='Region 0')
    ax.plot(t_s[:500], rates[:500, 1], color=col, lw=0.8, alpha=0.5, label='Region 1')
    ax.set_ylabel('Rate (Hz)', fontsize=9)
    ax.set_title(label, fontsize=10, fontweight='bold')
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Synthetic firing rates — first 2.5 s, two regions', fontsize=11)
plt.tight_layout()
plt.savefig('figures/01_raw_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Signal shape: {rates_cnt.shape} → (time={N_SAMPLES}, regions={N_REGIONS})")

---
## 2. Spectral Validity Check

Before running phase-coherence analysis, always verify that your target band contains a real spectral peak.  
The `phase_coherence_validity()` function runs three checks:  
1. **Spectral SNR** — in-band peak vs out-of-band median  
2. **Analytic amplitude** — mean envelope amplitude after narrowband filter  
3. **Kuramoto order** — global inter-regional synchrony level  

It will warn if anything looks problematic.

In [ ]:
import warnings

BANDPASS = (2.0, 80.0)  # Hz — covers full neural oscillation range for AdEx/MF

print("=== Spectral validity check ===")
for rates, label in [(rates_cnt, 'CNT'), (rates_mcs, 'MCS'), (rates_uws, 'UWS')]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        v = phase_coherence_validity(rates, dt_ms=DT_MS, bandpass_hz=BANDPASS)
    n_warn = len(caught)
    print(f"\n{label}:")
    print(f"  Kuramoto order : {v.kuramoto_order:.4f}")
    print(f"  Mean amplitude : {v.mean_amplitude.mean():.4f} (mean across regions)")
    print(f"  Has spectral peak: {v.has_spectral_peak.sum()}/{len(v.has_spectral_peak)} regions")
    if n_warn:
        for w in caught:
            print(f"  ⚠️  {w.message}")
    else:
        print("  ✅ No warnings")

# PSD plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, rates, label in zip(axes, [rates_cnt, rates_mcs, rates_uws], ['CNT', 'MCS', 'UWS']):
    result = psd_per_region(rates, dt_ms=DT_MS)
    mean_psd = result.power.mean(axis=1)  # average across regions
    ax.semilogy(result.frequencies, mean_psd, lw=1.5)
    ax.axvspan(BANDPASS[0], BANDPASS[1], alpha=0.12, color='green', label='Analysis band')
    ax.set_xlim(0, 100)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD')
    ax.set_title(label, fontweight='bold')
    dom_f = dominant_frequency(rates, dt_ms=DT_MS, bandpass_hz=BANDPASS)
    ax.set_title(f'{label}  (peak ~{dom_f.mean():.1f} Hz)', fontweight='bold')
    ax.legend(fontsize=8)
plt.suptitle('Mean PSD across regions — green = analysis band (2–80 Hz)', fontsize=11)
plt.tight_layout()
plt.savefig('figures/02_psd.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Brain States Analysis

### 3a. The `"firing_rate"` pipeline

What `pipeline="firing_rate"` does (in order):
1. **Discard transient** — drops first `transient_ms` samples (network settling)
2. **Z-score** — normalizes each region over time (ddof=1)
3. **Narrowband Butterworth filter** — here 2–80 Hz, 4th order — makes the signal single-band so Hilbert phases are meaningful
4. **Hilbert transform** — extracts the instantaneous phase envelope
5. **`angle()`** — instantaneous phase (no unwrapping — gamma is cyclostationary)
6. **`cos(φᵢ - φⱼ)`** — phase coherence between every region pair at each time step
7. **K-means** — clusters these pairwise snapshots into `n_states` recurring patterns

In [ ]:
import os
os.makedirs('figures', exist_ok=True)

N_STATES = 4
TRANSIENT_MS = 500.0  # discard first 500 ms (100 samples at 5 ms)

summaries = {}
for rates, label in [(rates_cnt, 'CNT'), (rates_mcs, 'MCS'), (rates_uws, 'UWS')]:
    summaries[label] = summarize_brain_states(
        rates,
        n_states=N_STATES,
        pipeline='firing_rate',
        bandpass_hz=BANDPASS,
        filter_order=4,
        dt_ms=DT_MS,
        transient_ms=TRANSIENT_MS,
        random_seed=42,
        n_init=10,
    )
    s = summaries[label]
    print(f"{label}: {s.labels.shape[0]} time points → {N_STATES} states")
    print(f"  Occupancy: {np.round(s.occupancy, 3)}")
    print(f"  Global sync mean: {s.global_synchrony.mean():.3f}")

In [ ]:
# ── Visualise brain state centroid matrices ──────────────────────────────────
fig, axes = plt.subplots(3, N_STATES, figsize=(4 * N_STATES, 9))

for row_idx, (label, col) in enumerate([('CNT', 'steelblue'), ('MCS', 'darkorange'), ('UWS', 'firebrick')]):
    s = summaries[label]
    mats = centers_to_matrices(s, n_regions=N_REGIONS)  # (n_states, N, N)
    for k in range(N_STATES):
        ax = axes[row_idx, k]
        im = ax.imshow(mats[k], cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
        occ = s.occupancy[k]
        ax.set_title(f'State {k+1}\nocc={occ:.2f}', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
        if k == 0:
            ax.set_ylabel(label, fontsize=11, fontweight='bold', color=col,
                          rotation=90, labelpad=5)

plt.suptitle(f'Brain-state centroids — cos(Δφ) patterns — {N_STATES} states\n(firing_rate pipeline, 2–80 Hz)', 
             fontsize=11)
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.5, label='cos(Δφ)')
plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.savefig('figures/03_brain_state_centroids.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Occupancy bar chart and global synchrony ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Occupancy
x = np.arange(N_STATES)
width = 0.25
colors = ['steelblue', 'darkorange', 'firebrick']
for i, (label, col) in enumerate([('CNT', 'steelblue'), ('MCS', 'darkorange'), ('UWS', 'firebrick')]):
    axes[0].bar(x + i * width, summaries[label].occupancy, width, label=label, color=col, alpha=0.85)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels([f'State {k+1}' for k in range(N_STATES)])
axes[0].set_ylabel('Occupancy (fraction of time)')
axes[0].set_title('State occupancy by condition')
axes[0].legend()

# Global synchrony over time
for label, col in [('CNT', 'steelblue'), ('MCS', 'darkorange'), ('UWS', 'firebrick')]:
    sync = summaries[label].global_synchrony
    t_sync = np.arange(len(sync)) * DT_MS / 1000.0
    # Rolling mean for readability
    window = min(50, len(sync) // 10)
    smoothed = np.convolve(sync, np.ones(window)/window, mode='same')
    axes[1].plot(t_sync, smoothed, color=col, lw=1.2, label=label)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('|R| (Kuramoto order)')
axes[1].set_title('Global phase synchrony over time (smoothed)')
axes[1].legend()

plt.suptitle('Brain-state summary statistics', fontsize=11)
plt.tight_layout()
plt.savefig('figures/04_brain_state_stats.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Transition matrices ───────────────────────────────────────────────────────
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (label, col) in zip(axes, [('CNT', 'Blues'), ('MCS', 'Oranges'), ('UWS', 'Reds')]):
    tm = summaries[label].transition_matrix
    sns.heatmap(tm, annot=True, fmt='.2f', cmap=col, ax=ax, vmin=0, vmax=1,
                xticklabels=[f'S{k+1}' for k in range(N_STATES)],
                yticklabels=[f'S{k+1}' for k in range(N_STATES)],
                cbar=False)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('To state')
    ax.set_ylabel('From state')
plt.suptitle('State transition matrices (row-normalised)', fontsize=11)
plt.tight_layout()
plt.savefig('figures/05_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

### 3b. Using real simulation outputs

Replace the synthetic data above with your real simulation output.  
The shapes expected by `summarize_brain_states` are:

```python
# AdEx SNN / mean-field (from prepare_population_rates, bin_width_ms=5.0)
rates = result.rate_exc_hz  # shape: (n_time_bins, n_regions) — or create from multiple results

summary = summarize_brain_states(
    rates,                      # (time, regions)
    n_states=5,
    pipeline='firing_rate',
    bandpass_hz=(2.0, 80.0),    # Full neural oscillation range for AdEx/MF
    filter_order=4,
    dt_ms=5.0,                  # Must match your actual bin width!
    transient_ms=500.0,         # Discard first 500 ms
    random_seed=42,
    n_init=10,
)

# TVB whole-brain — fast local oscillations
summary_fast = summarize_brain_states(
    tvb_rates,                  # shape (time, regions), raw dt=0.1 ms
    n_states=5,
    pipeline='firing_rate',
    bandpass_hz=(2.0, 80.0),
    filter_order=4,
    dt_ms=0.1,
    transient_ms=1000.0,
)

# BOLD (empirical or bold_from_firing_rates output)
summary_bold = summarize_brain_states(
    bold_signal,                # shape (time, regions)
    n_states=5,
    pipeline='standard',        # Brain-Act 04_02/04_04 parity
)
```

---
## 4. LZc — Lempel-Ziv Complexity

### What changed (the preprocessing fix)

LZc measures how algorithmically complex a binary sequence is. Before binarization, TVBSim applies:  
1. Mean subtraction per channel  
2. Linear detrend per channel  
3. Hilbert envelope extraction  
4. Threshold at mean envelope → binary  
5. Flatten to 1D (time-major) → LZ76 → normalize by shuffled surrogate  

The previous TVBToolkit version skipped steps 1–2. The fix adds `_preprocess()` before binarization.  
**This is already applied** in the current `lzc_multichannel()`.

### LZc on spontaneous firing rates vs BOLD

LZc on **spontaneous activity** is a valid measure — it does not require a TMS perturbation.  
It measures the algorithmic complexity of the ongoing signal: how diverse and non-repetitive the multichannel pattern is over time.  

**Important caveat**: for spontaneous firing rates, random/noisy signals can have *higher* LZc than structured ones.  
This means UWS (more random firing) may show higher LZc than CNT (more structured oscillations).  
This is not necessarily wrong — it reflects the signal statistics — but it may be opposite to the BOLD/EEG result.  
See Section 5 for a full discussion of why, and how PCI with perturbation fixes this.

In [ ]:
# ── Compute LZc for each condition ────────────────────────────────────────────
# We use a subwindow to keep computation fast (LZc is O(n log n) on the sequence)
# For real data you'd use the full signal.

TRANSIENT_BINS = int(TRANSIENT_MS / DT_MS)   # 100 samples
LZC_N_REPS = 5  # number of random seeds for estimating LZc variability

def compute_lzc_with_variability(rates, n_reps=5, transient_bins=100, seed_base=0):
    """Compute LZc with slight random variation from shuffled surrogate."""
    x = rates[transient_bins:]
    vals = []
    for i in range(n_reps):
        np.random.seed(seed_base + i)
        vals.append(lzc_multichannel(x))
    return np.array(vals)

lzc_results = {}
for rates, label in [(rates_cnt, 'CNT'), (rates_mcs, 'MCS'), (rates_uws, 'UWS')]:
    vals = compute_lzc_with_variability(rates, n_reps=LZC_N_REPS, transient_bins=TRANSIENT_BINS)
    lzc_results[label] = vals
    print(f"{label}: LZc = {vals.mean():.4f} ± {vals.std():.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
labels_plot = ['CNT', 'MCS', 'UWS']
colors_plot = ['steelblue', 'darkorange', 'firebrick']
means  = [lzc_results[l].mean() for l in labels_plot]
stds   = [lzc_results[l].std()  for l in labels_plot]

bars = ax.bar(labels_plot, means, yerr=stds, capsize=6,
              color=colors_plot, alpha=0.85, width=0.5, edgecolor='black')
ax.set_ylabel('Normalized LZc')
ax.set_title('Multichannel LZc — spontaneous firing rates\n(corrected: detrend + mean-subtract before binarization)', 
             fontsize=10)
ax.set_ylim(0, max(means) * 1.3)
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.005,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9)

ax.axhline(1.0, ls='--', color='gray', alpha=0.5, lw=1, label='Surrogate baseline (=1.0)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/06_lzc.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️  Note: UWS may show HIGHER LZc than CNT on spontaneous firing rates.")
print("   This is expected when UWS has more random/noisy activity (higher algorithmic complexity).")
print("   It is NOT a bug — see Section 5 for explanation and the correct approach using PCI with perturbation.")

---
## 5. PCI Casali-like

### The fundamental issue: PCI requires a perturbation

This is the most important thing to understand about PCI, and it directly explains why you are seeing **DoC > Control** ordering when applying PCI to spontaneous data.

**How Casali PCI actually works** (from the TVBSim code and the original paper):  

1. You apply a TMS pulse to the brain  
2. You record the neural response for the next ~300 ms  
3. The *pre-stimulus* baseline (the quiet period before the pulse) is used to normalize the signal  
4. PCI measures the **complexity of the perturbation response**, not spontaneous activity  

In a healthy brain, the TMS pulse propagates across many areas with rich, diverse dynamics → the response pattern is highly complex → high PCI.  
In UWS, the TMS pulse produces a simple stereotyped response (often just a slow wave spreading locally) → low complexity → low PCI.

**What happens with spontaneous data** (no perturbation):  

Without a real stimulus, there is no "response" to measure. You are just comparing one arbitrary window to another.  
The normalization threshold is set from the first half of the window ("pseudo-baseline"), and complexity is measured on the second half.  
In this case:
- **UWS simulations** often have *more random/noisy* activity (lower coupling → less structured oscillations)
- Random signals have high algorithmic complexity → high LZc → high PCI  
- **CNT simulations** have *more structured* oscillations (organized gamma) → lower LZc → lower PCI  
- **Result: DoC > Control — the opposite of the clinical finding**

This is not a bug in the code. It is a fundamental mismatch between what PCI measures and what you are giving it.

### The correct approach: perturbed simulations

To get PCI results that match the clinical DoC ordering, you need to:  
1. Run simulations with an explicit TMS-like perturbation (brief strong input to all or some regions)  
2. Cut a window of `t_analysis` ms before and after the stimulus onset  
3. Use `pci_casali_like()` with the correct `stimulation_index`

The cell below demonstrates this with synthetic perturbed data.

In [ ]:
# ── Demonstrate PCI with a simulated TMS perturbation ─────────────────────────
#
# We create a longer signal that has:
#   - 1500 ms of baseline activity (pre-stim)
#   - 1500 ms of post-stimulus response activity
#
# The stimulus response is simulated as:
#   CNT:  large, complex, spatially diverse response → many regions activated above threshold
#   MCS:  moderate response
#   UWS:  weak, simple, largely local response → few regions activated

T_PRESTIM_MS  = 1500.0   # ms before stimulus
T_POSTSTIM_MS = 1500.0   # ms after stimulus (t_analysis)
STIM_ONSET_MS = T_PRESTIM_MS

N_PRESTIM  = int(T_PRESTIM_MS  / DT_MS)   # 300 bins
N_POSTSTIM = int(T_POSTSTIM_MS / DT_MS)   # 300 bins
N_TOTAL    = N_PRESTIM + N_POSTSTIM        # 600 bins
STIM_BIN   = N_PRESTIM                     # index of stimulus onset

t_peri = np.arange(N_TOTAL) * DT_MS / 1000.0

def make_perturbed_signal(response_amplitude, response_coherence, noise_amp, seed=0):
    """Synthetic peri-stimulus signal.
    
    Pre-stim: background gamma + noise.
    Post-stim: background + TMS response (decaying oscillation, spatially diverse for CNT).
    """
    rng = np.random.default_rng(seed)
    
    # Background activity (same in both windows)
    t_all = np.arange(N_TOTAL) * DT_MS / 1000.0
    bg = 20 + 5 * np.stack(
        [np.sin(2 * np.pi * 40 * t_all + rng.uniform(0, 2*np.pi)) for _ in range(N_REGIONS)],
        axis=1
    ) + rng.normal(0, noise_amp, (N_TOTAL, N_REGIONS))
    
    # TMS response: brief, decaying oscillation starting at stimulus onset
    t_post = np.arange(N_POSTSTIM) * DT_MS / 1000.0  # 0 to 1.5 s
    decay = np.exp(-t_post / 0.3)  # 300 ms decay
    
    # Each region gets a different phase and amplitude in the response
    phase_offsets = rng.uniform(0, 2 * np.pi * (1 - response_coherence), N_REGIONS)
    response = response_amplitude * np.stack(
        [decay * np.sin(2 * np.pi * 12 * t_post + phi) for phi in phase_offsets],
        axis=1
    )
    
    signal = bg.copy()
    signal[N_PRESTIM:] += response
    return signal.clip(0)

# CNT: large complex response (high amplitude, diverse across regions)
peri_cnt = make_perturbed_signal(response_amplitude=15, response_coherence=0.1, noise_amp=2.0, seed=10)
# MCS: moderate response
peri_mcs = make_perturbed_signal(response_amplitude=8,  response_coherence=0.4, noise_amp=4.0, seed=11)
# UWS: weak simple response (low amplitude, few regions, more coherent/simple = lower complexity)
peri_uws = make_perturbed_signal(response_amplitude=3,  response_coherence=0.9, noise_amp=5.0, seed=12)

# ── Visualise peri-stimulus traces ───────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
for ax, sig, label, col in zip(
    axes,
    [peri_cnt, peri_mcs, peri_uws],
    ['CNT', 'MCS', 'UWS'],
    ['steelblue', 'darkorange', 'firebrick'],
):
    for r in range(N_REGIONS):
        ax.plot(t_peri, sig[:, r], color=col, lw=0.5, alpha=0.4)
    ax.axvline(t_peri[STIM_BIN], color='black', lw=2, ls='--', label='TMS stimulus')
    ax.set_ylabel('Rate (Hz)', fontsize=9)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Peri-stimulus firing rates — all regions — dashed line = TMS onset', fontsize=11)
plt.tight_layout()
plt.savefig('figures/07_peri_stim_traces.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Compute PCI with correct perturbation protocol ────────────────────────────
#
# pci_casali_like() expects:
#   x                : (time, channels) OR (channels, time) — auto-detected
#   stimulation_index: bin index of TMS onset
#   t_analysis_ms    : window size (ms) before AND after onset
#   dt_ms            : sampling interval

pci_perturbed = {}
for sig, label in [(peri_cnt, 'CNT'), (peri_mcs, 'MCS'), (peri_uws, 'UWS')]:
    debug = pci_casali_like(
        sig,
        stimulation_index=STIM_BIN,
        t_analysis_ms=T_POSTSTIM_MS,
        dt_ms=DT_MS,
        nshuffles=20,
        use_post_only=True,
        return_debug=True,
    )
    pci_perturbed[label] = debug
    print(f"{label}: PCI = {debug['pci']:.4f}  "
          f"(LZ={debug['lz']:.1f}, norm={debug['norm']:.1f}, "
          f"sparsity={debug['sparsity']:.3f}, entropy={debug['entropy']:.3f})")

print("\nExpected ordering: CNT > MCS > UWS (PCI decreases with DoC severity)")

In [ ]:
# ── Compare: spontaneous PCI (wrong) vs perturbed PCI (correct) ──────────────
# Also compute 'spontaneous PCI' using arbitrary mid-point as pseudo-stimulus
# to illustrate why this gives the wrong ordering.

PSEUDO_STIM_BIN = N_SAMPLES // 2  # arbitrary midpoint
PSEUDO_WINDOW_MS = 1500.0

pci_spontaneous = {}
for rates, label in [(rates_cnt, 'CNT'), (rates_mcs, 'MCS'), (rates_uws, 'UWS')]:
    try:
        val = pci_casali_like(
            rates[TRANSIENT_BINS:],          # discard transient
            stimulation_index=PSEUDO_STIM_BIN - TRANSIENT_BINS,
            t_analysis_ms=PSEUDO_WINDOW_MS,
            dt_ms=DT_MS,
            nshuffles=20,
            use_post_only=True,
        )
        pci_spontaneous[label] = val
    except ValueError as e:
        pci_spontaneous[label] = float('nan')
        print(f"{label}: could not compute spontaneous PCI — {e}")

# ── Bar chart comparison ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

lp = ['CNT', 'MCS', 'UWS']
cp = ['steelblue', 'darkorange', 'firebrick']

# Spontaneous (incorrect use)
vals_sp = [pci_spontaneous.get(l, float('nan')) for l in lp]
axes[0].bar(lp, vals_sp, color=cp, alpha=0.75, edgecolor='black')
axes[0].set_ylabel('PCI')
axes[0].set_title('PCI on SPONTANEOUS activity\n(no perturbation — INCORRECT use)', fontsize=10)
axes[0].set_ylim(0, max([v for v in vals_sp if not np.isnan(v)]) * 1.4)
for i, v in enumerate(vals_sp):
    if not np.isnan(v):
        axes[0].text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)
axes[0].text(0.5, 0.05, '⚠️  DoC may appear HIGHER than CNT\n(random signal = high complexity)',
            ha='center', transform=axes[0].transAxes, fontsize=9, color='darkred',
            bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.8))

# Perturbed (correct use)
vals_pt = [pci_perturbed[l]['pci'] for l in lp]
axes[1].bar(lp, vals_pt, color=cp, alpha=0.85, edgecolor='black')
axes[1].set_ylabel('PCI')
axes[1].set_title('PCI on PERTURBED activity\n(TMS-like stimulus — CORRECT use)', fontsize=10)
axes[1].set_ylim(0, max(vals_pt) * 1.4)
for i, v in enumerate(vals_pt):
    axes[1].text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)
axes[1].text(0.5, 0.05, '✅  CNT > MCS > UWS — correct DoC ordering',
            ha='center', transform=axes[1].transAxes, fontsize=9, color='darkgreen',
            bbox=dict(boxstyle='round', facecolor='honeydew', alpha=0.8))

plt.suptitle('PCI: spontaneous (incorrect) vs perturbed (correct)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/08_pci_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 5b. How to run PCI on your TVBToolkit simulations

```python
# Step 1: Run a simulation with an explicit stimulus
# In TVBSim, this means setting parameter_stimulus.stimtime.
# In TVBToolkit whole-brain sims, pass a stimulus object or add a brief
# strong Poisson input to the AdEx network at a specific time point.

# Step 2: Load the simulation output (time, regions)
rates = ...  # shape (n_time, n_regions), dt_ms known

# Step 3: Identify the stimulus onset bin
stim_onset_ms = 1000.0   # e.g. TMS at t=1000 ms
stim_bin = int(stim_onset_ms / dt_ms)

# Step 4: Choose the analysis window (ms before and after onset)
t_analysis_ms = 300.0    # 300 ms before AND after (TVBSim default)

# Step 5: Compute PCI
pci_val = pci_casali_like(
    rates,
    stimulation_index=stim_bin,
    t_analysis_ms=t_analysis_ms,
    dt_ms=dt_ms,
    nshuffles=10,            # TVBSim default
    percentile=100.0,        # TVBSim default  
    use_post_only=True,      # IMPORTANT: PCI is only on post-stimulus
)
```

**For multiple trials** (TVBSim-style, n_trials=5 seeds per PCI estimate):  
Stack trials along axis 0 and pass them all at once to get a more stable threshold:

```python
# Stack n_trials signals: shape (n_trials, n_regions, n_time)
signal_m = np.stack([rates_trial_0, rates_trial_1, ...], axis=0)
signal_binary = binarise_signals(
    signal_m.transpose(0, 1, 2),   # (trials, channels, time)
    t_stim=pre_stim_bins,
    nshuffles=10, percentile=100.0,
)
# Then compute PCI per trial on post-stim portion
```

---
## 6. Summary: Measure Selection Guide

| You have | You want to measure | Correct tool | Gives correct DoC ordering? |
|----------|--------------------|--------------|--------------------------|
| Spontaneous firing rates | Neural oscillation complexity | `lzc_multichannel()` | Depends on simulation — may invert |
| Spontaneous BOLD | Resting-state complexity | `lzc_multichannel()` | Often yes |
| Perturbed firing rates (TMS-like) | Perturbation response complexity | `pci_casali_like()` | ✅ Yes — matches clinical PCI |
| Spontaneous firing rates | Fast oscillation synchrony patterns | `summarize_brain_states(pipeline='firing_rate')` | Depends on interpretation |
| BOLD or HRF-convolved rates | Resting-state synchrony patterns | `summarize_brain_states(pipeline='standard')` | Yes — matches Brain-Act |

### Quick guidance on the DoC ordering problem

If you are getting **DoC > CNT** in PCI:  
→ You are almost certainly applying PCI to spontaneous activity without a real stimulus.  
→ The fix is to run simulations with an explicit TMS-like perturbation (brief strong input burst to all regions).  
→ In the interim, use `lzc_multichannel()` with care, noting the potential inversion.  

If you are getting **DoC > CNT** in LZc on spontaneous firing rates:  
→ This may be real: your DoC simulation parameters produce more random/less structured activity.  
→ Try converting to BOLD via `bold_from_firing_rates()` first, then computing LZc on the BOLD signal.  
→ BOLD LZc is more likely to show the expected ordering because the HRF integrates and smooths the activity, making structured oscillations look more orderly.

In [ ]:
# ── Final summary figure ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

lp = ['CNT', 'MCS', 'UWS']
cp = ['steelblue', 'darkorange', 'firebrick']

# LZc
axes[0].bar(lp, [lzc_results[l].mean() for l in lp],
            yerr=[lzc_results[l].std() for l in lp],
            color=cp, alpha=0.85, edgecolor='black', capsize=5)
axes[0].set_title('LZc (spontaneous)\n[may invert for firing rates]', fontsize=9)
axes[0].set_ylabel('Normalized LZc')

# Brain states — global synchrony
sync_means = [summaries[l].global_synchrony.mean() for l in lp]
axes[1].bar(lp, sync_means, color=cp, alpha=0.85, edgecolor='black')
axes[1].set_title('Global phase synchrony\n(brain states — firing_rate pipeline)', fontsize=9)
axes[1].set_ylabel('Mean |R| (Kuramoto)')

# PCI perturbed
axes[2].bar(lp, [pci_perturbed[l]['pci'] for l in lp], color=cp, alpha=0.85, edgecolor='black')
axes[2].set_title('PCI Casali-like (perturbed)\n[correct DoC ordering ✅]', fontsize=9)
axes[2].set_ylabel('PCI')

for ax in axes:
    ax.set_xlabel('Condition')

plt.suptitle('Three complexity / synchrony measures across DoC conditions', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/09_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("All figures saved to figures/")